# Generador de semillas arXiv para artículos CS

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
%%bash
# semilla.sh — celda bash para Google Colab
# Obtiene exactamente $META IDs de arXiv por categoría (categoría primaria), sin repetir
# los que ya están en semillas-ya-procesadas.txt ni los ya guardados en semilla.txt.
# Si una categoría no llega a la meta, vuelve a consultar con la fecha inicial 1 año antes,
# y así hasta llegar a la meta o hasta YEAR_MIN.

# ===================== AJUSTES =====================
META=10000                    # IDs exactos por categoría (antes: head -n XX)
ESPERADO=$((META))
MAX_RESULTS=5000              # max_results por consulta (el API de arXiv admite hasta 2000)
FECHA_INI=202611150000        # inicio del rango: AAAAMMDDhhmm
FECHA_FIN=202611152359        # fin del rango
YEAR_MIN=2007                 # no retroceder más allá de este año (arXiv empieza en 1991)
PAUSA=3                       # segundos entre consultas (arXiv pide ~3 s)
REINTENTOS_RED=3              # reintentos si arXiv responde vacío o con error

PROCESADAS=./semillas-ya-procesadas-11.txt     # ya existe; solo se lee, nunca se modifica
ETIQUETA=17nov2026                         # parte final del nombre de salida
SALIDA_TXT=semilla-${ESPERADO}-semilla-${ETIQUETA}.txt
SALIDA_CSV=semilla-${ESPERADO}-semilla-${ETIQUETA}.csv

# Alias: al final se cambia la categoría del .csv por su nombre cs.XX
declare -A ALIAS=( [math.NA]=cs.NA [eess.SY]=cs.SY )

#categorias=(cs.AI cs.CV cs.LG cs.CL cs.CR cs.AR cs.CC cs.CE cs.CG cs.CY cs.DB cs.DC cs.DL cs.DM cs.DS cs.ET cs.FL cs.GR cs.GT cs.HC cs.IR cs.IT cs.LO cs.MA cs.MM cs.MS cs.NE cs.NI cs.OH cs.OS cs.PF cs.PL cs.RO cs.SC cs.SD cs.SE cs.SI cs.GL math.NA eess.SY)

categorias=(cs.CL)
# categorias=(cs.IT)
# categorias=(cs.NE)
# categorias=(cs.LG)
# categorias=(cs.NA)

# Ajustes iniciales por categoría (las de difícil extracción y las que quieras).
# Las que no aparezcan usan FECHA_INI y MAX_RESULTS. Si alguna no llega a META,
# se retrocede 1 año la fecha inicial automáticamente hasta completarla.
declare -A INI_CAT=(
  [cs.GL]=199101010000
  [math.NA]=202101010000
  [eess.SY]=202101010000
)
declare -A MAX_CAT=(
  [cs.GL]=2000
  [math.NA]=2000
  [eess.SY]=2000
)
# ===================================================

TMPD=$(mktemp -d)
trap 'rm -rf "$TMPD"' EXIT
cd "$(dirname "$0")" 2>/dev/null || true   # en Colab: la carpeta actual de la celda
if [[ ! -s $PROCESADAS ]]; then
  echo "ERROR: no encuentro $PROCESADAS en $(pwd)" >&2; exit 1
fi
[[ -e $SALIDA_TXT ]] || : > "$SALIDA_TXT"   # solo se crean si no existen; nunca se vacían
[[ -e $SALIDA_CSV ]] || : > "$SALIDA_CSV"

# Lista de exclusión: IDs ya procesados + ya guardados (sin versión, sin "arXiv:", sin \r)
normalizar() { sed -E 's/\r//; s/^[[:space:]]*(arXiv:)?//I; s/[,[:space:]].*$//; s/v[0-9]+$//; /^$/d'; }
cat "$PROCESADAS" "$SALIDA_TXT" | normalizar | sort -u > "$TMPD/excluir"
echo "Excluyendo $(wc -l < "$TMPD/excluir") IDs ya conocidos." >&2

# Consulta arXiv y escribe los IDs (sin versión, p. ej. 2301.12345) de la categoría primaria $1
consultar() {
  local cat=$1 ini=$2 max=$3 r resp
  for (( r = 1; r <= REINTENTOS_RED; r++ )); do
    resp=$(curl -A "Mozilla/5.0" -s --max-time 120 -G "https://export.arxiv.org/api/query" \
            --data-urlencode "search_query=cat:${cat} AND submittedDate:[${ini} TO ${FECHA_FIN}]" \
            -d "sortBy=relevance" -d "sortOrder=descending" \
            -d "start=0" -d "max_results=${max}")
    if grep -q '<entry>' <<< "$resp" || grep -q 'totalResults[^>]*>0<' <<< "$resp"; then
      grep -Po '<id>http://arxiv.org/abs/\K[^<]+|<arxiv:primary_category\s+term="\K[^"]+' <<< "$resp" \
        | paste -d, - - | grep -E ",${cat//./\\.}\$" | cut -d, -f1 \
        | sed -E 's/v[0-9]+$//'
      return 0
    fi
    echo "    (respuesta vacía, reintento $r/$REINTENTOS_RED)" >&2
    sleep $(( PAUSA * r * 2 ))
  done
  return 1
}

total=${#categorias[@]}
i=0
for cat in "${categorias[@]}"; do
  i=$(( i + 1 ))
  ini=${INI_CAT[$cat]:-$FECHA_INI}
  max=${MAX_CAT[$cat]:-$MAX_RESULTS}
  pool=$TMPD/pool
  : > "$pool"

  while :; do
    echo "[$i/$total] ${cat}: desde ${ini:0:4}, max_results=${max}..." >&2
    consultar "$cat" "$ini" "$max" > "$TMPD/nuevos" || echo "    sin respuesta de arXiv" >&2
    # Quitar los ya conocidos y unir con lo acumulado en intentos anteriores
    grep -vxFf "$TMPD/excluir" "$TMPD/nuevos" | cat - "$pool" | sort -u > "$TMPD/pool2"
    mv "$TMPD/pool2" "$pool"
    n=$(wc -l < "$pool")
    echo "    -> ${n}/${META} IDs nuevos" >&2
    sleep "$PAUSA"

    (( n >= META )) && break
    year=$(( ${ini:0:4} - 1 ))
    if (( year < YEAR_MIN )); then
      echo "    AVISO: ${cat} solo tiene ${n} IDs nuevos hasta ${YEAR_MIN}; se guardan esos." >&2
      break
    fi
    ini=${year}${ini:4}
  done

  # Elegir META al azar, guardar y sumarlos a la exclusión
  shuf -n "$META" "$pool" | sort -r > "$TMPD/elegidos"
  sed "s/\$/,${ALIAS[$cat]:-$cat}/" "$TMPD/elegidos" >> "$SALIDA_CSV"
  cat "$TMPD/elegidos" >> "$SALIDA_TXT"
  cat "$TMPD/elegidos" "$TMPD/excluir" | sort -u > "$TMPD/ex2" && mv "$TMPD/ex2" "$TMPD/excluir"
  echo "    guardados $(wc -l < "$TMPD/elegidos") IDs de ${cat}" >&2
done

echo "Listo: $(wc -l < "$SALIDA_TXT") IDs en $SALIDA_TXT y $SALIDA_CSV" >&2
cut -d, -f2 "$SALIDA_CSV" | sort | uniq -c | awk '$1 != '"$META"' {print "  AVISO: " $2 " tiene " $1 " IDs"}' >&2

In [ ]:
# https://info.arxiv.org/help/find/index.html
# https://arxiv.org/list/archive.SUBCATEGORY/YYYY

# https://arxiv.org/list/cs.GL/2026
# https://arxiv.org/list/cs.GL/2025
# https://arxiv.org/list/cs.GL/2024
# https://arxiv.org/list/cs.GL/2023
# https://arxiv.org/list/cs.GL/2022
# https://arxiv.org/list/cs.GL/2021
# https://arxiv.org/list/cs.GL/2020
# https://arxiv.org/list/cs.GL/2019
# https://arxiv.org/list/cs.GL/2018
# https://arxiv.org/list/cs.GL/2017
# https://arxiv.org/list/cs.GL/2016
# https://arxiv.org/list/cs.GL/2015
# https://arxiv.org/list/cs.GL/2014
# https://arxiv.org/list/cs.GL/2013
# https://arxiv.org/list/cs.GL/2012
# https://arxiv.org/list/cs.GL/2011
# https://arxiv.org/list/cs.GL/2010


# for year in {2026..1990}; do
#      echo "Fetching cs.GL papers for $year..." >&2 \
#      curl -A "Mozilla/5.0" -s "https://arxiv.org/list/cs.GL/${year}?show=100"
#          | grep -Po 'title="Abstract" id="\K[^"]+'
#      sleep 1
#  done > csGL.txt
